In [1]:
import pandas as pd
import numpy as np
df=pd.read_csv('01_Trucktrafficflow.csv')


In [2]:
# Your list of allowed regions
regions = [
    "Delfzijl en omgeving", "Oost-Groningen", "Overig Groningen", "Zuidoost-Friesland",
    "Noord-Friesland", "Zuidwest-Friesland", "Noord-Drenthe", "Zuidoost-Drenthe",
    "Zuidwest-Drenthe", "Noord-Overijssel", "Zuidwest-Overijssel", "Twente",
    "Veluwe", "Zuidwest-Gelderland", "Achterhoek", "Arnhem/Nijmegen", "Flevoland",
    "Kop van Noord-Holland", "IJmond", "Zaanstreek", "Het Gooi en Vechtstreek",
    "Alkmaar en omgeving", "Agglomeratie Haarlem", "Groot-Amsterdam",
    "Zeeuwsch-Vlaanderen", "Overig Zeeland", "Utrecht", "Agglomeratie ’s-Gravenhage",
    "Delft en Westland", "Agglomeratie Leiden en Bollenstreek", "Zuidoost-Zuid-Holland",
    "Oost-Zuid-Holland", "Groot-Rijnmond", "West-Noord-Brabant", "Zuidoost-Noord-Brabant",
    "Midden-Noord-Brabant", "Noordoost-Noord-Brabant", "Noord-Limburg",
    "Midden-Limburg", "Zuid-Limburg"
]

# Ensure df is already loaded before this step
# Example:
# df = pd.read_csv("your_file.csv")

# Filter rows based on region list
filtered_df = df.loc[
    df["Name_origin_region"].isin(regions),
    ["Name_origin_region", "Name_destination_region", "Total_distance", "Traffic_flow_tons_2019", "Edge_path_E_road"]
]
filtered_df.head()

,Name_origin_region,Name_destination_region,Total_distance,Traffic_flow_tons_2019,Edge_path_E_road
1087774,Oost-Groningen,Mittelburgenland,1145.0,17.0,"[1034974, 1008535, 1008552, 1000001, 1008851, ..."
1087775,Oost-Groningen,Nordburgenland,1121.0,51.0,"[1008555, 2500448, 1008540, 1008817, 1008826, ..."
1087776,Oost-Groningen,Sudburgenland,1243.0,17.0,"[1008563, 1310426, 1009011, 1035597, 1008566, ..."
1087777,Oost-Groningen,Mostviertel-Eisenwurzen,1085.0,187.0,"[1008584, 1008595, 1008621, 1035056, 1008618, ..."
1087778,Oost-Groningen,Niederosterreich-Sud,1164.0,221.0,"[1008654, 1008675, 1000017, 1035287, 1008678, ..."


In [2]:


# -----------------------------
# Reference points (lat, lon)
# -----------------------------
points = np.array([
    [4.5967195, 51.832247,],  # Kijfhoek
    [4.6058241, 51.674466,],  # Moerdijk
    [3.6938758, 51.448118,],  # Sloe
    [4.8997765, 52.379106,],  # Amsterdam
])

point_names = ["Kijfhoek", "Moerdijk", "Sloe", "Amsterdam"]

# -----------------------------
# Load region data
# -----------------------------
Reg = pd.read_csv("02_NUTS-3-Regions.csv")

regions = [
    "Delfzijl en omgeving", "Oost-Groningen", "Overig Groningen", "Zuidoost-Friesland",
    "Noord-Friesland", "Zuidwest-Friesland", "Noord-Drenthe", "Zuidoost-Drenthe",
    "Zuidwest-Drenthe", "Noord-Overijssel", "Zuidwest-Overijssel", "Twente",
    "Veluwe", "Zuidwest-Gelderland", "Achterhoek", "Arnhem/Nijmegen", "Flevoland",
    "Kop van Noord-Holland", "IJmond", "Zaanstreek", "Het Gooi en Vechtstreek",
    "Alkmaar en omgeving", "Agglomeratie Haarlem", "Groot-Amsterdam",
    "Zeeuwsch-Vlaanderen", "Overig Zeeland", "Utrecht", "Agglomeratie ’s-Gravenhage",
    "Delft en Westland", "Agglomeratie Leiden en Bollenstreek", "Zuidoost-Zuid-Holland",
    "Oost-Zuid-Holland", "Groot-Rijnmond", "West-Noord-Brabant", "Zuidoost-Noord-Brabant",
    "Midden-Noord-Brabant", "Noordoost-Noord-Brabant", "Noord-Limburg",
    "Midden-Limburg", "Zuid-Limburg"
]

Reg_df = Reg.loc[
    Reg["Name"].isin(regions),
    ["Name", "Geometric_center", "Geometric_center_X", "Geometric_center_Y"]
].copy()

# -----------------------------
# Haversine function
# -----------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

# -----------------------------
# Compute distances
# -----------------------------
distances = []

for lat, lon in points:
    dist = haversine(
        Reg_df["Geometric_center_X"].values,  # lat
        Reg_df["Geometric_center_Y"].values,  # lon
        lat,
        lon
    )
    distances.append(dist)

distances = np.array(distances)  # shape: (num_points, num_regions)

# -----------------------------
# Find closest point
# -----------------------------
closest_idx = distances.argmin(axis=0)

Reg_df["min_distance_km"] = distances.min(axis=0)
Reg_df["closest_point"] = [point_names[i] for i in closest_idx]

# -----------------------------
# Result
# -----------------------------
Reg_df

,Name,Geometric_center,Geometric_center_X,Geometric_center_Y,min_distance_km,closest_point
327,Zeeuwsch-Vlaanderen,POINT (3.79663 51.304),3.79663,51.3040,19.653463,Sloe
328,Overig Zeeland,POINT (3.85662 51.5743),3.85662,51.5743,22.879835,Sloe
341,Delft en Westland,POINT (4.26768 51.9879),4.26768,51.9879,40.452661,Kijfhoek
342,Groot-Rijnmond,POINT (4.29228 51.8525),4.29228,51.8525,33.926505,Kijfhoek
349,Agglomeratie Leiden en Bollenstreek,POINT (4.51273 52.2044),4.51273,52.2044,42.294825,Kijfhoek
353,West-Noord-Brabant,POINT (4.56229 51.5605),4.56229,51.5605,13.527677,Moerdijk
354,Agglomeratie Haarlem,POINT (4.62653 52.3733),4.62653,52.3733,30.390435,Amsterdam
355,IJmond,POINT (4.65719 52.5017),4.65719,52.5017,30.201897,Amsterdam
358,Oost-Zuid-Holland,POINT (4.71145 52.0697),4.71145,52.0697,29.245699,Kijfhoek
359,Alkmaar en omgeving,POINT (4.7683 52.6563),4.76830,52.6563,34.014908,Amsterdam
